# 🧱 支线 5：LLM 架构改进 — 从「能跑」到「好训」

> **前情回顾**：在 Part 4 我们搭了一个 MiniGPT，用的是经典 Transformer 配方：
> - `x + Attention(x)` 做完 Attention 再做残差，最后 LayerNorm（**Post-Norm**）
> - FFN 用 ReLU 激活（**ReLU FFN**）
> - 归一化用 LayerNorm（**LayerNorm**）
>
> **但真正的 LLM（LLaMA、GPT-4、DeepSeek）用的是另一套配方。**
>
> 本 Part 目标：**把 Part 4 的 Transformer Block 一点点升级成现代 LLM 同款，理解每一步「为什么」。**

三个改进，每改一处都问三个问题：
1. 旧的是什么？怎么算的？
2. 旧的有什么问题？
3. 新的怎么算？为什么更好？

**全部用极小例子手算，保证每一步都能跟上。**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### 0. 先回顾：Part 4 的 Block 长什么样？在这里动手改

先把你已经熟悉的 Part 4 Block 贴出来，我们就在它上面一个一个改。

In [ ]:
# === 这就是 Part 4 的版本，我们接下来的「改造对象」 ===

class FeedForward_Old(nn.Module):
    """原始 FFN：两层 Linear，中间 ReLU"""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class TransformerBlock_Old(nn.Module):
    """Post-Norm + LayerNorm + ReLU FFN（Part 4 版本）"""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn = FeedForward_Old(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)  # ← 普通 LayerNorm
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Post-Norm: 先算子层，再 +残差，最后 Norm
        x = self.norm1(x + self.attention(x, x, x, need_weights=False)[0])
        x = self.norm2(x + self.ffn(x))
        return x

print("✅ 这是 Part 4 的版本。接下来我们逐个升级。")
print("升级路线: LayerNorm→RMSNorm → ReLU→SwiGLU → Post-Norm→Pre-Norm")

---

### 1. 改进一：LayerNorm → RMSNorm

#### 1.1 LayerNorm 到底在算什么？— 用 4 个数字手算一遍

假设一个 token 的向量是 `[1, 3, 5, 7]`（4 维，和我们 Part 4 的 d_model=4 一样）。

**LayerNorm 做的事**：把这个向量的均值变成 0，标准差变成 1。

就像给一群学生调分——不管原始分数多高多低，调完之后平均分 0 分，分数分散程度统一。

具体步骤：
```
1. 算均值  μ = (x₁ + x₂ + x₃ + x₄) / 4
2. 算方差  σ² = ((x₁-μ)² + (x₂-μ)² + (x₃-μ)² + (x₄-μ)²) / 4
3. 算标准差 σ = √σ²
4. 归一化 x' = (x - μ) / σ
5. 缩放   y = γ × x' + β
```

γ 和 β 是可学习的参数（让模型自己决定调完之后「放多大」「往哪偏」）。

下面用 `[1, 3, 5, 7]` 手算一遍：

In [ ]:
# === LayerNorm 手工计算 ===
print("=== LayerNorm 手算：输入 x = [1, 3, 5, 7] ===")
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# Step 1: 均值
mu = x.mean()
print(f"Step 1 — 均值 μ = (1+3+5+7)/4 = {mu:.1f}")

# Step 2: 方差（这里除以 N，不是 N-1）
var = torch.mean((x - mu) ** 2)
print(f"Step 2 — 方差 σ² = ((1-4)²+(3-4)²+(5-4)²+(7-4)²)/4")
print(f"        = (9 + 1 + 1 + 9)/4 = {var:.1f}")

# Step 3: 标准差
sigma = torch.sqrt(var)
print(f"Step 3 — 标准差 σ = √{var:.1f} = {sigma:.4f}")

# Step 4: 归一化
x_norm = (x - mu) / sigma
print(f"Step 4 — 归一化: (x - 4)/{sigma:.4f}")
for i, (xi, xni) in enumerate(zip(x.tolist(), x_norm.tolist())):
    print(f"         x[{i}] = ({xi:.1f} - 4) / {sigma:.4f} = {xni: .4f}")

print(f"         归一化后: {[f'{v:.4f}' for v in x_norm.tolist()]}")
print(f"         均值: {x_norm.mean():.4f} (=0), 标准差: {x_norm.std(unbiased=False):.4f} (=1)")

# Step 5: 缩放（假设 γ=[1,1,1,1], β=[0,0,0,0]）
# 初始时 γ 全 1，β 全 0，所以输出就是 x_norm
print(f"Step 5 — 缩放: γ=1, β=0 时输出 = 归一化结果")
print(f"         γ 和 β 是随着训练学出来的，让模型自己决定怎么调")

#### 1.2 LayerNorm 的问题：多算了一个不需要的东西

注意 LayerNorm 算了两个统计量：
- **μ（均值）**：把数据中心移到 0
- **σ（标准差）**：把数据分散程度统一

有人做了实验：**如果把「去均值」这一步去掉，只做缩放，效果会变差吗？**

答案：**不会。** 因为后面的线性变换（W_Q, W_K 等）本身就可以学会处理有均值的数据。

那还留着均值计算干嘛？去掉它，每次就能省一半的计算。

#### 1.3 RMSNorm：只做缩放

RMSNorm 的核心公式：

```
RMS(x) = √( (x₁² + x₂² + x₃² + x₄²) / 4 )

RMSNorm(x) = x / RMS(x) × γ
```

**对比**：
```
LayerNorm: (x - μ) / σ  × γ + β    ← μ 和 σ 两个统计量
RMSNorm:    x / RMS(x) × γ          ← 只有 RMS 一个统计量
```

省掉的是：不用算均值，不用算 (x-μ) 的偏差平方（只需要算 x²）。

用同一个 `[1, 3, 5, 7]` 手算 RMSNorm：

In [ ]:
# === RMSNorm 手工计算 ===
print("=== RMSNorm 手算：输入 x = [1, 3, 5, 7] ===")
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# 唯一的一步：算 RMS
mean_sq = torch.mean(x ** 2)
rms = torch.sqrt(mean_sq)

print(f"Step 1 — 平方均值 = (1²+3²+5²+7²)/4")
print(f"          = (1+9+25+49)/4")
print(f"          = {84/4:.1f}")
print(f"    RMS = √{mean_sq:.1f} = {rms:.4f}")
print()

# RMSNorm 输出
x_rmsnorm = x / rms
print(f"Step 2 — RMSNorm = x / {rms:.4f}")
for i, (xi, xri) in enumerate(zip(x.tolist(), x_rmsnorm.tolist())):
    print(f"         x[{i}] = {xi:.1f} / {rms:.4f} = {xri:.4f}")

print(f"\nRMSNorm 结果: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}")

# 验证：RMSNorm 后的 RMS 应该是 1
rms_after = torch.sqrt(torch.mean(x_rmsnorm ** 2))
print(f"\n验证 — 归一化后的 RMS = {rms_after:.4f} (应该 = 1) ✅")
print(f"        注意：RMSNorm 后的均值 ≠ 0（均值 = {x_rmsnorm.mean():.4f}）")
print(f"        这就是和 LayerNorm 的区别——不去均值！")
print()

# 对比
ln = nn.LayerNorm(4, elementwise_affine=False)  # 关掉 γ,β 方便对比
x_ln = ln(x)
print(f"LayerNorm 结果: {[f'{v:.4f}' for v in x_ln.tolist()]}  均值={x_ln.mean():.4f}  std={x_ln.std(unbiased=False):.4f}")
print(f"RMSNorm  结果: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}  均值={x_rmsnorm.mean():.4f}  RMS={rms_after:.4f}")
print(f"\n→ 数值不同，但都起到了「标准化」的作用")
print(f"→ LayerNorm 强制均值=0，RMSNorm 不强制（但效果一样好）")

In [ ]:
# === RMSNorm 完整实现 ===

class RMSNorm(nn.Module):
    """
    RMSNorm: 只做缩放，不做中心化
    
    公式: y = x / RMS(x) × γ
    RMS(x) = √(mean(x²) + ε)
    
    - 没有 β（bias），因为不需要补偿去均值造成的偏移
    - ε 是为了防止除以 0（比如输入全 0）
    """
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))  # 只有 γ
    
    def forward(self, x):
        # x: [batch, seq_len, d_model]
        # 沿最后一维算 RMS
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.gamma


# 测试
d_model = 8
rmsn = RMSNorm(d_model)
ln = nn.LayerNorm(d_model)

batch = torch.randn(2, 4, d_model) * 3  # 模拟 2 个 batch, 4 个 token, 8 维
out_rms = rmsn(batch)
out_ln = ln(batch)

print("=== RMSNorm 实现验证 ===")
print(f"输入形状: {batch.shape}")
print(f"RMSNorm 输出形状: {out_rms.shape}")
print(f"输出 RMS（应该 ≈ 1）: {torch.sqrt(torch.mean(out_rms**2, dim=-1))[0].tolist()}")
print()

# 参数量对比
ln_params = sum(p.numel() for p in ln.parameters())
rmsn_params = sum(p.numel() for p in rmsn.parameters())
print(f"LayerNorm 参数: γ({d_model}) + β({d_model}) = {ln_params}")
print(f"RMSNorm  参数: γ({d_model}) = {rmsn_params}")
print(f"→ 参数减半，但这不是重点——重点是计算时少算了均值")

---

### 2. 改进二：ReLU → SwiGLU

#### 2.1 先看原始 FFN 里 ReLU 做了什么

Part 4 的 FFN 公式：

```
FFN(x) = W₂ · ReLU(W₁ · x)

ReLU 做的事：负数 → 0，正数 → 原样通过
```

用一个具体的向量手算看看：

In [ ]:
# === ReLU 手算 ===
print("=== ReLU 对每个数字做了什么 ===")
print()

# 模拟 W₁·x 的输出（扩展到 8 维）
hidden = torch.tensor([0.5, -2.0, 3.0, -0.1, 0.0, -5.0, 1.5, -0.01])

print(f"输入 (W₁·x 的结果): {[f'{v:.2f}' for v in hidden.tolist()]}")
print()

relu_out = F.relu(hidden)
print("ReLU 做了什么：")
for i, (h, r) in enumerate(zip(hidden.tolist(), relu_out.tolist())):
    action = "保留" if h >= 0 else "→ 0 (杀死)"
    print(f"  位置 {i}: {h:6.2f} → {r:6.2f}  {action}")

print(f"\n问题：8 个位置中有 {sum(1 for h in hidden if h < 0)} 个被直接杀死了")
print(f"      其中 -0.1 和 -0.01 只是「稍微负了一点」，也被杀了")
print(f"      → ReLU 太粗暴：负的全部不要")

#### 2.2 门控的直觉：编辑 vs 总编

想象你在写一篇文章，然后交给编辑部审稿：

- **ReLU 方式（一个编辑）**：编辑看完每一段，要么全文通过（正），要么全文删除（负）。一段话如果有 90% 好 + 10% 有瑕疵 → 整个删了，可惜。

- **门控方式（两个编辑）**：
  - 编辑 A：负责理解内容（信息通道）
  - 编辑 B：负责打分（门控通道）——这段有多重要？给 0~1 之间的分数
  - 最终输出 = A 的理解 × B 的分数

效果：如果一段话「稍微有点问题」，B 给 0.3 分，它还是能通过一部分——而不是完全消失。

公式就长这样：

```
SwiGLU(x) = (W_up · x)  ⊙  Swish(W_gate · x)
              ↑ 信息通道       ↑ 门控通道
              
Swish(a) = a × sigmoid(a) = a / (1 + e^(-a))
```

**Swish 是什么？** 它是一个平滑的「软开关」——不像 ReLU 那样一刀切。

In [ ]:
# === ReLU vs Swish 手算对比 ===
print("=== 激活函数对比：ReLU vs Swish ===")
print()

x_vals = [-3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]

print(f"{'x':>8s}  {'ReLU(x)':>10s}  {'Swish(x)':>10s}  {'说明'}")
print("-" * 52)

for x in x_vals:
    relu = max(0, x)
    swish = x / (1 + math.exp(-x))  # x * sigmoid(x)
    note = ""
    if x < 0:
        note = f"ReLU 杀死, Swish 保留 {swish:.4f}"
    elif x == 0:
        note = "ReLU 卡在 0, Swish 平滑"
    else:
        note = "两者都通过"
    print(f"{x:>8.1f}  {relu:>10.1f}  {swish:>10.4f}  {note}")

print()
print("关键观察：")
print("  1. 正数区: Swish ≈ ReLU（略小一点），行为相似")
print("  2. 负数区: ReLU = 0（彻底杀死），Swish 保留一个小负值")
print("  3. x=0 处: ReLU 不可导（数学上），Swish 光滑可导")
print()
print("为什么「小负值」重要？")
print("  梯度 = ∂loss/∂x，而 ∂Swish/∂x 在负数区不为 0")
print("  → 即使这个神经元当前「不太激活」，梯度仍然可以流过")
print("  → ReLU 的神经元可能「永久死亡」（一旦进入负数区，梯度=0，再也回不来）")

In [ ]:
# === 手动计算 SwiGLU 的一步 ===
print("=== SwiGLU 手算 ===")
print()

# 用极小例子：d_model=4, 模拟一条信息
d_model = 4
x = torch.tensor([[1.0, -0.5, 2.0, 0.3]])  # 一个 token

# 模拟三个权重矩阵的简单版本（手工构造便于观察）
# 实际中这些是 nn.Linear 的权重
torch.manual_seed(1)
W_up = nn.Linear(d_model, d_model, bias=False)
W_gate = nn.Linear(d_model, d_model, bias=False)
W_down = nn.Linear(d_model, d_model, bias=False)

# 信息通道
up = W_up(x)
# 门控通道：先线性变换，再过 Swish
gate = F.silu(W_gate(x))  # silu = Swish
# 门控结果 = 信息 × 门
gated = up * gate
# 输出投影
out = W_down(gated)

print(f"输入 x: {x.tolist()}")
print()
print(f"信息通道 (W_up·x):     {[f'{v:.3f}' for v in up[0].tolist()]}")
print(f"门控通道 Swish(W_gate·x): {[f'{v:.3f}' for v in gate[0].tolist()]}")
print(f"门控后 (up × gate):     {[f'{v:.3f}' for v in gated[0].tolist()]}")
print(f"输出 (W_down·gated):    {[f'{v:.3f}' for v in out[0].tolist()]}")
print()
print("门控的效果: 如果 gate 某个位置 ≈ 0 → 信息通道对应位置被「关门」→ 信息传不过去")
print("           如果 gate 某个位置 ≈ 1 → 信息通道对应位置「开门」→ 信息完全通过")
print("           如果 gate 某个位置 = 0.3 → 信息只通过 30%")

In [ ]:
# === 完整 SwiGLU FFN 实现 ===

class FeedForward_SwiGLU(nn.Module):
    """
    SwiGLU FFN（LLaMA / DeepSeek 同款）
    
    公式: FFN(x) = W_down · ( Swish(W_gate·x) ⊙ (W_up·x) )
    
    三个权重矩阵：
    - W_gate: 门控投影 (d_model → d_ff)
    - W_up:   信息投影 (d_model → d_ff)
    - W_down: 输出投影 (d_ff → d_model)
    
    注意：原始 FFN 有 2 个矩阵，SwiGLU 有 3 个。为了保持参数量一致，
    d_ff 要调小：原始 d_ff=4d，SwiGLU d_ff ≈ 8/3 d
    """
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            # 3 * d_model * d_ff ≈ 2 * d_model * 4d_model
            # → d_ff ≈ 8/3 * d_model
            d_ff = int(8 / 3 * d_model)
            d_ff = ((d_ff + 255) // 256) * 256  # 取整到 256 的倍数
            d_ff = max(d_ff, d_model)  # 至少和 d_model 一样大
        
        self.W_gate = nn.Linear(d_model, d_ff, bias=False)
        self.W_up = nn.Linear(d_model, d_ff, bias=False)
        self.W_down = nn.Linear(d_ff, d_model, bias=False)
    
    def forward(self, x):
        return self.W_down(F.silu(self.W_gate(x)) * self.W_up(x))


# 参数量对比
d_model = 512
ffn_old = FeedForward_Old(d_model)
ffn_new = FeedForward_SwiGLU(d_model)

old_p = sum(p.numel() for p in ffn_old.parameters())
new_p = sum(p.numel() for p in ffn_new.parameters())

print("=== SwiGLU vs ReLU FFN 参数量对比 ===")
print(f"d_model = {d_model}")
print()
print(f"ReLU FFN:  W1({d_model}×{4*d_model}) + W2({4*d_model}×{d_model})")
print(f"           = {d_model*4*d_model:,} + {4*d_model*d_model:,}")
print(f"           = {old_p:,} 个参数")
print()
d_ff_s = ((int(8/3*d_model) + 255) // 256) * 256
print(f"SwiGLU FFN: W_gate({d_model}×{d_ff_s}) + W_up({d_model}×{d_ff_s}) + W_down({d_ff_s}×{d_model})")
print(f"           = {d_model*d_ff_s:,} + {d_model*d_ff_s:,} + {d_ff_s*d_model:,}")
print(f"           = {new_p:,} 个参数")
print()
print(f"参数量比: {new_p/old_p:.2f}x (≈1 即可)")
print(f"→ 参数差不多，但 SwiGLU 效果好很多")

---

### 3. 改进三：Post-Norm → Pre-Norm

#### 3.1 先搞清 Post-Norm 的「Post」是什么意思

Part 4 的 Block 里，写法是这样：

```python
x = x + attention(x)   # 残差：把输入和 Attention 输出加起来
x = norm(x)            # LayerNorm：标准化
```

这里 LayerNorm 在 Attention **之后**，所以叫 **Post-Norm**。

完整的计算图（Post-Norm）：

```
  x ──→ Attention ──→ + ──→ LayerNorm ──→ 输出
  │                    ↑
  └────────────────────┘  (残差连接)
```

注意：残差连接加到 Attention 的输出上，然后两者一起被 LayerNorm。
**这意味着残差路径经过了 LayerNorm！**

#### 3.2 这有什么问题？

回忆 Part 4，我们说过残差连接是「梯度高速公路」——梯度可以通过残差路径直接回传，不被 Attention 和 FFN 衰减。

但现在，在 Post-Norm 里，**这条高速公路上多了一个收费站——LayerNorm**。

LayerNorm 会缩放梯度，这意味着经过 LayerNorm 后，梯度的大小可能被改变。

在 4 层网络里这不是问题。但在 40 层、80 层网络里，每一层都经过一次 LayerNorm → 梯度的缩放效应会累积 → 可能爆炸或消失。

#### 3.3 Pre-Norm：收费站搬到辅路上

Pre-Norm 的做法：把 LayerNorm 搬到 Attention **之前**：

```python
x = x + attention(norm(x))   # 先 Norm，再做 Attention
```

计算图（Pre-Norm）：

```
  x ──→ LayerNorm ──→ Attention ──→ + ──→ 输出
  │                                   ↑
  └───────────────────────────────────┘  (残差连接，不经过 Norm！)
```

**关键变化**：残差连接绕过了 LayerNorm！梯度在高速公路上没有任何收费站。

用两个极小的网络对比：

In [ ]:
# === Post-Norm vs Pre-Norm：用手算看梯度流动 ===
print("=== Post-Norm vs Pre-Norm：梯度流动对比 ===")
print()

# 搭两个简化的深层网络（用 FFN 代替 Attention，专注看 Norm 位置的影响）
d_model = 4
num_layers = 8  # 先用 8 层看效果

class Deep_PostNorm(nn.Module):
    """Post-Norm: x = Norm(x + FFN(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
    
    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = norm(x + F.relu(layer(x)))
        return x

class Deep_PreNorm(nn.Module):
    """Pre-Norm: x = x + FFN(Norm(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
    
    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = x + F.relu(layer(norm(x)))
        return x

# 相同初始化
torch.manual_seed(42)
model_post = Deep_PostNorm(d_model, num_layers)
torch.manual_seed(42)
model_pre = Deep_PreNorm(d_model, num_layers)

# 随机输入
x = torch.randn(2, d_model)
target = torch.randn(2, d_model)

# Forward + Backward
loss_post = F.mse_loss(model_post(x), target)
loss_post.backward()

loss_pre = F.mse_loss(model_pre(x), target)
loss_pre.backward()

# 看每一层的梯度范数
print(f"网络深度: {num_layers} 层")
print()
print(f"{'层':>6s}  {'Post-Norm 梯度':>16s}  {'Pre-Norm 梯度':>16s}")
print("-" * 42)

for i in range(num_layers):
    grad_post = model_post.layers[i].weight.grad.norm().item()
    grad_pre = model_pre.layers[i].weight.grad.norm().item()
    print(f"Layer {i+1}:  {grad_post:>16.6f}  {grad_pre:>16.6f}")

# 最重要：看第 1 层的梯度（最底层）
grad_post_first = model_post.layers[0].weight.grad.norm().item()
grad_pre_first = model_pre.layers[0].weight.grad.norm().item()

print()
print(f"最关键——最底层（Layer 1）梯度对比:")
print(f"  Post-Norm Layer 1 梯度: {grad_post_first:.6f}")
print(f"  Pre-Norm  Layer 1 梯度: {grad_pre_first:.6f}")
print(f"  Pre-Norm / Post-Norm = {grad_pre_first/grad_post_first:.2f}x")
print()
print("→ Pre-Norm 底层梯度更大，因为残差路径绕过了所有 LayerNorm")
print("→ 在 80 层网络里，这个差别会大到 Post-Norm 根本训不动")

#### 3.4 用做饭类比总结

- **Post-Norm**：先炒菜，再尝味道 → 如果炒糊了，调整也没用（梯度已经被 Norm 截断了）
- **Pre-Norm**：先尝调料确认味道，再炒 → 炒的过程中信号始终稳定（残差路径畅通）

所以在深层网络中，Pre-Norm 的训练稳定得多——它可以用更大的学习率，更少的 warmup。

---

### 4. 拼起来！现代 LLaMA-style Block

三个改进全部到位，把它们一次组装：

```
LLaMA Block = Pre-Norm (RMSNorm) + Attention + Pre-Norm (RMSNorm) + SwiGLU FFN

  x
  │
  ├─→ RMSNorm → Attention ─→ +  ← 残差（不经过 Norm！）
  │                           │
  ├───────────────────────────┘
  │
  ├─→ RMSNorm → SwiGLU FFN ─→ +  ← 残差（不经过 Norm！）
  │                            │
  └────────────────────────────┘
  ↓
 输出
```

In [ ]:
class LLaMABlock(nn.Module):
    """
    现代 LLM 的 Transformer Block
    
    三大升级 vs Part 4:
    1. LayerNorm → RMSNorm
    2. Post-Norm → Pre-Norm
    3. ReLU FFN → SwiGLU FFN
    """
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        # Pre-Norm: RMSNorm 在 Attention 前面
        self.norm_attn = RMSNorm(d_model)
        # Multi-Head Self-Attention（和 Part 4 一样）
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        # Pre-Norm: RMSNorm 在 FFN 前面
        self.norm_ffn = RMSNorm(d_model)
        # SwiGLU FFN
        self.ffn = FeedForward_SwiGLU(d_model, d_ff)
    
    def forward(self, x, mask=None):
        # 注意顺序：先 Norm，再子层，再残差
        h = self.norm_attn(x)
        attn_out, _ = self.attention(h, h, h, attn_mask=mask, need_weights=False)
        x = x + attn_out  # 残差：绕过 Norm！
        
        h = self.norm_ffn(x)
        x = x + self.ffn(h)  # 残差：绕过 Norm！
        
        return x

# 测试
d_model, num_heads = 8, 2
block_new = LLaMABlock(d_model, num_heads)

test_x = torch.randn(1, 5, d_model)
causal_mask = torch.triu(torch.ones(5, 5) * float('-inf'), diagonal=1)

out_new = block_new(test_x, causal_mask)
print("=== LLaMA Block 测试 ===")
print(f"输入形状: {test_x.shape}")
print(f"输出形状: {out_new.shape}  ← 不变！")
print(f"但里面的组件全部升级了:")
print(f"  ✓ RMSNorm 替代 LayerNorm")
print(f"  ✓ Pre-Norm 替代 Post-Norm")
print(f"  ✓ SwiGLU 替代 ReLU FFN")

### 5. 三代 Transformer Block 演进全景图

```
2017  原始 Transformer (Vaswani et al.)
      Post-Norm + LayerNorm + ReLU FFN
      └→ 能跑，但深层不好训（原始论文只用了 6 层）

2019  GPT-2 / GPT-3
      Pre-Norm + LayerNorm + GELU FFN
      └→ 改了 Norm 位置，能堆到 96 层了
          GELU 比 ReLU 好一点，但还是不够好

2023  LLaMA 2 / 3, DeepSeek, Qwen, Mistral...
      Pre-Norm + RMSNorm + SwiGLU FFN
      └→ 三个改进全用上，训练又快又稳效果又好
```

如果要给三个改进的重要性排序：
1. **Pre-Norm**（最重要——决定了能不能训深层）
2. **SwiGLU**（效果明显——门控机制让 FFN 更聪明）
3. **RMSNorm**（锦上添花——省了计算，但换了也不影响大局）

---

## 架构改进小结

确认你已经搞懂了这些（按顺序检查）：

1. ✅ LayerNorm = (x - μ) / σ × γ + β，算两个统计量
2. ✅ RMSNorm = x / RMS(x) × γ，只算一个统计量——去掉了去均值
3. ✅ RMSNorm 省的计算：不用算 μ，不用算 (x-μ)²（直接用 x²）
4. ✅ ReLU 的问题：负数直接杀 → 梯度断掉，神经元可能死亡
5. ✅ SwiGLU = 信息通道 (W_up) × 门控通道 (Swish(W_gate))→ 选择性通过
6. ✅ Swish 平滑且负数区有非零梯度 → 不会「杀死」神经元
7. ✅ Post-Norm = 子层+T残加→Norm，残差路径经过 Norm
8. ✅ Pre-Norm = Norm→子层→+残差，残差路径绕过 Norm
9. ✅ Pre-Norm 好在哪：梯度高速公路无收费站 → 深层也能训
10. ✅ LLaMA Block = Pre-Norm + RMSNorm + SwiGLU（现代 LLM 标准配方）

**一句话总结**：三个改进解决三个不同层面的问题——Pre-Norm 解决「能不能训」(稳定性)、SwiGLU 解决「训得好不好」(效果)、RMSNorm 解决「训得快不快」(效率)。